# Solar Active-Region Detection — Kaggle training run

**One cell to run.** Run the code cell below. It sets everything up (repo, Python environment, data download) and starts max-speed training on the GPU.

Before you run it, check the notebook settings (right panel, **Settings**):

1. **Internet: ON** — required for downloading the solar data and installing packages. (Kaggle only shows this switch for phone-verified accounts.)
2. **GPU: T4 x2 or P100** — any free GPU works; the run picks it up automatically.

Kaggle stops a notebook session after ~12 hours. **That is fine**: just run the same cell again in the same notebook and it picks up exactly where it stopped (downloaded frames and model checkpoints are kept on the 20 GB working disk).

In [ ]:
%%bashset -xexport HOME=/kaggle/workcd /kaggle/workif [ -d SOALR/.git ]; then    git -C SOALR pull -q || trueelse    git clone -q -b arena/01a04247-soalr-active-region-detection \        https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \    || { mkdir -p SOALR && wget -qO /tmp/repo.tgz \        https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \        && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }fiif [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# Reuse Kaggle's preinstalled CUDA torch (saves a ~2 GB download):if [ ! -x .venv/bin/python ]; then    python3 -m venv --system-site-packages .venvfi.venv/bin/python -m pip install -q -r requirements.txt# Kaggle only lets you DOWNLOAD files from /kaggle/output (zip available in the# notebook's "Output" tab after each session ends). This watcher keeps a fresh# copy of the latest model + log there every 5 minutes, so whatever session# dies, the Output tab holds the newest model:mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/work/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/work/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# MAX-SPEED preset, tuned for Kaggle's 20 GB working disk:#   MIN_FREE_GB=4        (the default 40 GB reserve would disable ALL downloads on Kaggle)#   DOWNLOAD_WORKERS=4   (each in-flight frame is a ~570 MB temp file)#   MAX_TOTAL_FRAMES=1500 (disk is the bottleneck; the run self-limits below it)CHANNELS="aia94 aia131 aia1600 aia171 aia193 aia211 aia304 aia335 hmi_m hmi_bx hmi_by hmi_bz hmi_v" \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=1500 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=4 \CPU_HEADROOM=2 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- **Pre-flight** lines like `[ OK ] mask index train.csv: 70,823 available frames` — all three data sources (mask index, mask archive, S3 frames) are probed before anything big happens. If one says `[FAIL]`, that line tells you exactly why downloads would not work.
- `Downloading 200 frames with 4 parallel workers ...`, then `OK <timestamp> -> N tiles` per frame.
- `[stream] batch_size=...` and per-epoch lines; checkpoints save to `/kaggle/work/solar_results/arpil/continuous/`.

## If the session dies (12-hour cap or a crash)

Run the code cell above again — same notebook. Nothing is lost; it resumes downloads and training from the last saved checkpoint.